In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 1. Load Data

In [ ]:
# Load dataset
df = pd.read_csv(
    '../data/raw/2025_(JAN-MAR).csv',
    parse_dates=['Published Date', 'Closing Date', 'Award Date']
)

print(f"Loaded: {len(df)} rows")
df.head()

Loaded: 299 rows


,Procuring Entity,Region,Province,City/Municipality,Government Branch,PE Organization Type,PE Organization Type (Grouped),Bid Reference No.,Notice Title,Classification,Procurement Mode,Business Category,Source of Funds,Trade Agreement,Approved Budget of the Contract (ABC),Published Date,Closing Date,Area of Delivery,Contract Duration,Calendar Type,Lot Type,Item Code,Item/Lot Name,Item/Lot Description,Quantity,Unit of Measure,Item Budget,Bid Notice Status,Award Reference No.,Award Title,UNSPSC Code,UNSPSC Description,Published Date (Award),Award Date,Contract Amount,Award Notice Status,Notice to Proceed Date,Contract Effectivity Date,Contract End Date,Awardee Organization Name,Country of Awardee,Region of Awardee,Province of Awardee,City/Municipality of Awardee,Awardee Size,Awardee Joint Venture
0,PHILIPPINE COMMISSION ON SPORTS SCUBA DIVING,NCR,Metro Manila,Makati City,Executive,Attached Agency,National Government Agencies (NGA),3827,Conduct of BOT Regular/Special Meeting,Goods,Small Value Procurement,Travel facilitation,Regular Agency Fund (01000000),Implementing Rules and Regulations,151500.0,2025-01-16,2025-01-20,Metro Manila,3,days,Single Lot,90121502,Travel agencies,Procurement of Travel and Tour Services for\r\...,1,Lot,0.0,Closed,1365,Conduct of BOT Regular/Special Meeting,90121502,Travel agencies,2025-02-21,2025-01-20,151500.0,Posted/Published,NaN,NaN,NaN,EAS - ELLEN'S TRAVEL & TOURS,Philippines,Region IV-B,Palawan,Puerto Princesa City,Micro,NaN
1,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Metro Manila,Manila,Executive,Department,National Government Agencies (NGA),3843,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,Consulting Services,Public Bidding,Professional engineering services,Regular Agency Fund (01000000),Implementing Rules and Regulations,571629232.5,2025-01-17,2025-02-27,NaN,44,months,Single Lot,81101505,Structural engineering,Structural engineering,1,Lot,0.0,Closed,1669,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,81101508,Architectural engineering,2025-07-15,2025-07-15,535456140.0,Posted/Published,NaN,NaN,NaN,WOODFIELDS ENGINEERS COMPANY,Philippines,Region IV-A,Rizal,Rodriguez (Montalban),Medium,NaN
2,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Metro Manila,Manila,Executive,Department,National Government Agencies (NGA),3843,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,Consulting Services,Public Bidding,Professional engineering services,Regular Agency Fund (01000000),Implementing Rules and Regulations,571629232.5,2025-01-17,2025-02-27,NaN,44,months,Single Lot,81101505,Structural engineering,Structural engineering,1,Lot,0.0,Closed,1669,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,81101526,Quantity surveying service,2025-07-15,2025-07-15,535456140.0,Posted/Published,NaN,NaN,NaN,WOODFIELDS ENGINEERS COMPANY,Philippines,Region IV-A,Rizal,Rodriguez (Montalban),Medium,NaN
3,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Metro Manila,Manila,Executive,Department,National Government Agencies (NGA),3843,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,Consulting Services,Public Bidding,Professional engineering services,Regular Agency Fund (01000000),Implementing Rules and Regulations,571629232.5,2025-01-17,2025-02-27,NaN,44,months,Single Lot,81101505,Structural engineering,Structural engineering,1,Lot,0.0,Closed,1669,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,81101510,Highway engineering,2025-07-15,2025-07-15,535456140.0,Posted/Published,NaN,NaN,NaN,WOODFIELDS ENGINEERS COMPANY,Philippines,Region IV-A,Rizal,Rodriguez (Montalban),Medium,NaN
4,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Metro Manila,Manila,Executive,Department,National Government Agencies (NGA),3843,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,Consulting Services,Public Bidding,Professional engineering services,Regular Agency Fund (01000000),Implementing Rules and Regulations,571629232.5,2025-01-17,2025-02-27,NaN,44,months,Single Lot,81101505,Structural engineering,Structural engineering,1,Lot,0.0

## 2. Handle Duplicates

Same bid appears multiple times with different UNSPSC codes. We'll aggregate to contract level.

In [3]:
# Group by bid reference and aggregate
df_clean = df.groupby('Bid Reference No.').agg({
    'Procuring Entity': 'first',
    'Region': 'first',
    'Classification': 'first',
    'Procurement Mode': 'first',
    'Approved Budget of the Contract (ABC)': 'first',
    'Published Date': 'first',
    'Closing Date': 'first',
    'Award Date': 'first',
    'Contract Amount': 'first',
    'Contract Duration': 'first',
    'Awardee Organization Name': 'first',
    'Awardee Size': 'first',
    'Notice Title': 'first'
}).reset_index()

print(f"After deduplication: {len(df_clean)} unique contracts")
df_clean.head()

After deduplication: 59 unique contracts


,Bid Reference No.,Procuring Entity,Region,Classification,Procurement Mode,Approved Budget of the Contract (ABC),Published Date,Closing Date,Award Date,Contract Amount,Contract Duration,Awardee Organization Name,Awardee Size,Notice Title
0,3827,PHILIPPINE COMMISSION ON SPORTS SCUBA DIVING,NCR,Goods,Small Value Procurement,151500.0,2025-01-16,2025-01-20,2025-01-20,1.515000e+05,3,EAS - ELLEN'S TRAVEL & TOURS,Micro,Conduct of BOT Regular/Special Meeting
1,3843,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Consulting Services,Public Bidding,571629232.5,2025-01-17,2025-02-27,2025-07-15,5.354561e+08,44,WOODFIELDS ENGINEERS COMPANY,Medium,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...
2,4001,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Consulting Services,Small Value Procurement,220000.0,2025-02-04,2025-02-10,2025-03-04,2.000000e+05,11,DR RACQUEL YAMBALLA QUILANG,Micro,Procurement of Service for 2025 PhilRice Medic...
3,4002,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Goods,Small Value Procurement,130500.0,2025-02-04,2025-02-07,2025-02-10,1.122000e+05,3,CHEF-DE-CUISINE FOOD STORE,Micro,Catering Services for the 2024 DA-PhilRice Ann...
4,4004,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Civil Works - Infra Project,Public Bidding,1510005.0,2025-02-05,2025-02-25,2025-03-20,1.320034e+06,120,G.R. ERCILLA CONSTRUCTION,Micro,One (1) lot Replacement of Floor Tiles at Rice...


## 3. Create Derived Features

In [4]:
# Price variance
df_clean['price_variance_pct'] = (
    (df_clean['Contract Amount'] - df_clean['Approved Budget of the Contract (ABC)']) / 
    df_clean['Approved Budget of the Contract (ABC)'] * 100
)

# Bidding duration (days)
df_clean['bidding_duration_days'] = (
    df_clean['Closing Date'] - df_clean['Published Date']
).dt.days

# Award speed (days)
df_clean['award_speed_days'] = (
    df_clean['Award Date'] - df_clean['Closing Date']
).dt.days

# Winner frequency
winner_counts = df_clean['Awardee Organization Name'].value_counts()
df_clean['winner_frequency'] = df_clean['Awardee Organization Name'].map(winner_counts)

print("\nDerived features created:")
print(df_clean[['price_variance_pct', 'bidding_duration_days', 'award_speed_days', 'winner_frequency']].describe())


Derived features created:
       price_variance_pct  bidding_duration_days  award_speed_days  \
count           59.000000              59.000000         59.000000   
mean           -13.360061               9.677966         24.271186   
std             17.593100               9.366928         24.257122   
min            -72.659506               3.000000          0.000000   
25%            -14.394914               3.000000          9.000000   
50%             -6.956553               5.000000         21.000000   
75%             -2.147206              20.000000         30.000000   
max              0.000000              41.000000        138.000000   

       winner_frequency  
count         59.000000  
mean           1.305085  
std            0.564900  
min            1.000000  
25%            1.000000  
50%            1.000000  
75%            1.500000  
max            3.000000  


## 4. Handle Missing Values

In [5]:
# Check missing values in key columns
print("Missing values:")
print(df_clean[['price_variance_pct', 'bidding_duration_days', 'award_speed_days']].isnull().sum())

# Drop rows with missing critical features
df_clean = df_clean.dropna(subset=['price_variance_pct', 'bidding_duration_days', 'award_speed_days'])

# Replace inf values with NaN then drop
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=['price_variance_pct'])

print(f"\nAfter cleaning: {len(df_clean)} rows")

Missing values:
price_variance_pct       0
bidding_duration_days    0
award_speed_days         0
dtype: int64

After cleaning: 59 rows


## 5. Save Processed Data

In [6]:
# Save to processed folder
output_path = Path('../data/processed')
output_path.mkdir(exist_ok=True)

df_clean.to_csv(output_path / 'procurement_processed.csv', index=False)
print(f"✅ Saved to {output_path / 'procurement_processed.csv'}")

# Preview final data
df_clean.head()

✅ Saved to ../data/processed/procurement_processed.csv


,Bid Reference No.,Procuring Entity,Region,Classification,Procurement Mode,Approved Budget of the Contract (ABC),Published Date,Closing Date,Award Date,Contract Amount,Contract Duration,Awardee Organization Name,Awardee Size,Notice Title,price_variance_pct,bidding_duration_days,award_speed_days,winner_frequency
0,3827,PHILIPPINE COMMISSION ON SPORTS SCUBA DIVING,NCR,Goods,Small Value Procurement,151500.0,2025-01-16,2025-01-20,2025-01-20,1.515000e+05,3,EAS - ELLEN'S TRAVEL & TOURS,Micro,Conduct of BOT Regular/Special Meeting,0.000000,4,0,1
1,3843,DEPARTMENT OF PUBLIC WORKS AND HIGHWAYS - MAIN,NCR,Consulting Services,Public Bidding,571629232.5,2025-01-17,2025-02-27,2025-07-15,5.354561e+08,44,WOODFIELDS ENGINEERS COMPANY,Medium,CONSULTING SERVICES FOR THE CONSTRUCTION SUPER...,-6.328069,41,138,1
2,4001,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Consulting Services,Small Value Procurement,220000.0,2025-02-04,2025-02-10,2025-03-04,2.000000e+05,11,DR RACQUEL YAMBALLA QUILANG,Micro,Procurement of Service for 2025 PhilRice Medic...,-9.090909,6,22,1
3,4002,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Goods,Small Value Procurement,130500.0,2025-02-04,2025-02-07,2025-02-10,1.122000e+05,3,CHEF-DE-CUISINE FOOD STORE,Micro,Catering Services for the 2024 DA-PhilRice Ann...,-14.022989,3,3,2
4,4004,PHILIPPINE RICE RESEARCH INSTITUTE,Region III,Civil Works - Infra Project,Public Bidding,1510005.0,2025-02-05,2025-02-25,2025-03-20,1.320034e+06,120,G.R. ERCILLA CONSTRUCTION,Micro,One (1) lot Replacement of Floor Tiles at Rice...,-12.580836,20,23,1


In [7]:
# Summary stats
print("\nFinal Dataset Summary:")
print(f"Total contracts: {len(df_clean)}")
print(f"\nFeatures available for modeling:")
print(df_clean.select_dtypes(include=[np.number]).columns.tolist())


Final Dataset Summary:
Total contracts: 59

Features available for modeling:
['Bid Reference No.', 'Approved Budget of the Contract (ABC)', 'Contract Amount', 'Contract Duration', 'price_variance_pct', 'bidding_duration_days', 'award_speed_days', 'winner_frequency']
